In [ ]:
# One-time installs (run in notebook cell if not installed yet)
!pip install sqlalchemy mysql-connector-python pymysql plotly pandas statsmodels

import os
import calendar

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.api as sm

from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Use notebook renderer for Plotly
pio.renderers.default = "vscode"

# -------- AWS RDS connection config --------
db_user  = os.getenv("STAGING_DB_USER", "admin")
db_pass  = os.getenv("STAGING_DB_PASSWORD")
db_host  = os.getenv(
    "STAGING_DB_HOST",
    "your-db-host.example.com",
)
db_port  = int(os.getenv("STAGING_DB_PORT", "3306"))

# 👉 IMPORTANT: point to the DW schema, not staging
db_name  = os.getenv("STAGING_DB_NAME", "warehouse")

# (Optional) keep if you still want a default table name somewhere
db_table = os.getenv("STAGING_DB_TABLE", "Fact_Label")

# Build SQLAlchemy URL for AWS RDS (MySQL + PyMySQL)
db_url = URL.create(
    drivername="mysql+pymysql",
    username=db_user,
    password=db_pass,
    host=db_host,
    port=db_port,
    database=db_name,
)

# Create SQLAlchemy engine pointing to AWS RDS warehouse
engine = create_engine(db_url)

# --- Example: test query (you can change this) ---
# df = pd.read_sql("SELECT * FROM Fact_Label LIMIT 5", con=engine)
# display(df.head())

## 1. Load Gobest Cab DW Star Schema Tables

In this section, we load the three core tables from the `warehouse` schema into pandas:

- `Dim_Driver` – driver and car attributes with outlier flags and MV scores  
- `Dim_Sensor` – one row per trip (booking_id) with engineered trip-level sensor features  
- `Fact_Label` – one row per trip (booking_id) storing safety labels and links to the dimensions  

This will give us a working DataFrame for each table to run EDA on ranges, distributions, class balance, and outliers.


In [ ]:
# 1. Load DW tables into pandas DataFrames

dim_driver_df = pd.read_sql("SELECT * FROM Dim_Driver", con=engine)
dim_sensor_df = pd.read_sql("SELECT * FROM Dim_Sensor", con=engine)
fact_label_df = pd.read_sql("SELECT * FROM Fact_Label", con=engine)

print("Shapes:")
print("  Dim_Driver:", dim_driver_df.shape)
print("  Dim_Sensor:", dim_sensor_df.shape)
print("  Fact_Label:", fact_label_df.shape)

display(dim_driver_df.head())
display(dim_sensor_df.head())
display(fact_label_df.head())


## 2. Basic Info and Missing Values

Before diving into distributions, we quickly inspect:

- Column types for each table (`info()`).
- Missing values per column.  

This helps us understand which fields are numeric vs categorical and highlights any data quality issues that may affect downstream analysis.


In [ ]:
# 2. Basic info and missing values for each table

for name, df in [
    ("Dim_Driver", dim_driver_df),
    ("Dim_Sensor", dim_sensor_df),
    ("Fact_Label", fact_label_df),
]:
    print(f"\n=== {name}: info() ===")
    df.info()

    print(f"\n=== {name}: missing values ===")
    missing_summary = df.isna().sum().sort_values(ascending=False)
    display(missing_summary)


## 3. Identify Numeric vs Categorical Columns

We now separate numeric and non-numeric columns for each table.  

This split helps us:
- Run summary stats (min, max, mean, percentiles) for numeric columns.
- Run frequency tables for categorical columns like gender or outlier flags.


In [ ]:
# 3. Identify numeric vs categorical columns for each table

def split_columns(df):
    num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = df.select_dtypes(exclude=["number"]).columns.tolist()
    return num_cols, cat_cols

dim_driver_num_cols, dim_driver_cat_cols = split_columns(dim_driver_df)
dim_sensor_num_cols, dim_sensor_cat_cols = split_columns(dim_sensor_df)
fact_label_num_cols, fact_label_cat_cols = split_columns(fact_label_df)

print("Dim_Driver numeric:", dim_driver_num_cols)
print("Dim_Driver categorical:", dim_driver_cat_cols)

print("\nDim_Sensor numeric:", dim_sensor_num_cols)
print("Dim_Sensor categorical:", dim_sensor_cat_cols)

print("\nFact_Label numeric:", fact_label_num_cols)
print("Fact_Label categorical:", fact_label_cat_cols)


## 4. Summary Statistics (Ranges) for Each Table

We compute `.describe()` for numeric columns to inspect:

- Minimum and maximum values (range)
- Central tendency (mean/median)
- Spread (standard deviation, quartiles)

This gives a quick overview of value ranges and helps detect obviously wrong values (e.g. negative durations or impossible years).


In [ ]:
# 4.1 Summary stats for Dim_Driver numeric columns
dim_driver_desc = dim_driver_df[dim_driver_num_cols].describe().T
print("=== Dim_Driver: numeric summary ===")
display(dim_driver_desc)

# 4.2 Summary stats for Dim_Sensor numeric columns
dim_sensor_desc = dim_sensor_df[dim_sensor_num_cols].describe().T
print("=== Dim_Sensor: numeric summary ===")
display(dim_sensor_desc)

# 4.3 Summary stats for Fact_Label numeric columns
fact_label_desc = fact_label_df[fact_label_num_cols].describe().T
print("=== Fact_Label: numeric summary ===")
display(fact_label_desc)


## 5. Dim_Driver – Driver & Car Profile Analysis

`Dim_Driver` stores both driver and car attributes.  
Here we focus on:

- Distributions of key numeric variables:  
  - `no_of_years_driving_exp` (driving experience)  
  - `rating` (driver rating)  
  - `car_model_year` (car age)  
- Outlier flags and multivariate outlier indicator (`mv_is_outlier`).

We will:
1. Plot histograms and boxplots for the main numeric fields.
2. Check frequency of outlier flags and multivariate scores.


In [ ]:
# ================================
# 5. Dim_Driver – Driver & Car Profile Analysis
# ================================

# Helper: plot histogram + boxplot for a numeric column, optionally split by a label column
def plot_numeric_distribution(df, col, color_col=None, nbins=50, title_prefix=""):
    """
    Plot histogram and boxplot for a numeric column.
    If color_col is provided, histogram is split by that column.
    """
    if col not in df.columns:
        print(f"[WARN] Column {col} not found in DataFrame.")
        return

    # Histogram
    if color_col is not None and color_col in df.columns:
        fig_hist = px.histogram(
            df,
            x=col,
            color=color_col,
            nbins=nbins,
            opacity=0.7,
            marginal="rug",
            title=f"{title_prefix}Distribution of {col} by {color_col}"
        )
    else:
        fig_hist = px.histogram(
            df,
            x=col,
            nbins=nbins,
            opacity=0.7,
            marginal="rug",
            title=f"{title_prefix}Distribution of {col}"
        )

    fig_hist.show()

    # Boxplot
    if color_col is not None and color_col in df.columns:
        fig_box = px.box(
            df,
            x=color_col,
            y=col,
            points="outliers",
            title=f"{title_prefix}Boxplot of {col} by {color_col}"
        )
    else:
        fig_box = px.box(
            df,
            y=col,
            points="outliers",
            title=f"{title_prefix}Boxplot of {col}"
        )

    fig_box.show()

In [ ]:
# 5.1 Key numeric distributions in Dim_Driver
driver_numeric_focus = ["no_of_years_driving_exp", "rating", "car_model_year"]

for col in driver_numeric_focus:
    if col in dim_driver_df.columns:
        print(f"\n=== Dim_Driver: {col} ===")
        plot_numeric_distribution(dim_driver_df, col=col)

# 5.2 Gender distribution (categorical)
if "gender" in dim_driver_df.columns:
    print("\n=== Dim_Driver: Gender distribution ===")

    # Make a proper DataFrame with clear column names
    gender_counts = (
        dim_driver_df["gender"]
        .value_counts(dropna=False)
        .rename_axis("gender")        # index -> 'gender'
        .reset_index(name="count")    # value_counts -> 'count'
    )
    display(gender_counts)

    fig_gender = px.bar(
        gender_counts,
        x="gender",
        y="count",
        labels={"gender": "Gender", "count": "Number of drivers"},
        title="Gender Distribution"
    )
    fig_gender.show()

# 5.3 Outlier flags and MV outliers
outlier_flag_cols = [
    c for c in dim_driver_df.columns
    if c.endswith("__is_uni_outlier") or c in ["mv_is_outlier"]
]

print("\n=== Dim_Driver: Outlier flag distributions ===")
for col in outlier_flag_cols:
    print(f"\n{col}:")

    vc_df = (
        dim_driver_df[col]
        .value_counts(dropna=False)
        .rename_axis(col)           # index -> col name
        .reset_index(name="count")  # counts -> 'count'
    )
    display(vc_df)

    fig_flag = px.bar(
        vc_df,
        x=col,
        y="count",
        labels={col: col, "count": "Number of drivers"},
        title=f"Distribution of {col}"
    )
    fig_flag.show()

In [ ]:
import plotly.io as pio

print("Default renderer BEFORE:", pio.renderers.default)

# If you're in Jupyter Notebook / JupyterLab:
pio.renderers.default = "notebook"        # or "notebook_connected"

# If you're in VS Code Jupyter:
# pio.renderers.default = "vscode"

# Fallback that always works:
# pio.renderers.default = "browser"

print("Default renderer AFTER:", pio.renderers.default)



In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "vscode"

fig = px.histogram(dim_driver_df, x="rating", nbins=20, title="Test Rating Histogram")
fig.show()



### 6.1 Class imbalance for `label` column

We compute the class counts and percentages for the `label` column  
(0 = Safe, 1 = Dangerous) and visualise them using a bar chart.


In [ ]:
# ================================
# 6. Fact_Label – Class Imbalance & Integrity Checks
# ================================

# 6.1 Class imbalance for label column
label_col = "label"  # 0 = Safe, 1 = Dangerous

if label_col in fact_label_df.columns:
    # Build a clean DataFrame with counts + percentage
    class_dist = (
        fact_label_df[label_col]
        .value_counts(dropna=False)
        .rename_axis(label_col)        # index -> 'label'
        .reset_index(name="count")     # counts -> 'count'
    )
    class_dist["percentage"] = (
        class_dist["count"] / class_dist["count"].sum() * 100
    ).round(2)

    print("=== Fact_Label: class distribution ===")
    display(class_dist)

    # Human-readable label names for plotting
    df_plot = class_dist.copy()
    df_plot["label_str"] = df_plot[label_col].map({0: "Safe", 1: "Dangerous"}).fillna(
        df_plot[label_col].astype(str)
    )

    fig_label = px.bar(
        df_plot,
        x="label_str",
        y="count",
        text="percentage",
        labels={"label_str": "Class", "count": "Number of trips"},
        title=f"Class Distribution – {label_col}"
    )
    fig_label.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig_label.update_layout(yaxis_title="Count", xaxis_title="Class")
    fig_label.show()
else:
    print(f"[WARN] label_col '{label_col}' not found in Fact_Label.")

# 6.2 Booking_id uniqueness check
print("\n=== Fact_Label: booking_id uniqueness check ===")
dup_booking = fact_label_df["booking_id"].duplicated().sum()
print("Duplicated booking_id rows:", dup_booking)

# 6.3 Foreign key coverage: non-null driver_key and sensor_key
print("\n=== Fact_Label: FK non-null coverage ===")
fk_cols = ["driver_key", "sensor_key"]
for col in fk_cols:
    total = fact_label_df.shape[0]
    non_null = fact_label_df[col].notna().sum()
    print(f"{col}: {non_null}/{total} non-null ({non_null/total*100:.2f}%)")

## 7. Join Fact_Label with Dimensions (Star-Join View)

To understand how driver and sensor features relate to safety labels, we create a **denormalised view**:

`fact_star_df` = Fact_Label ⨝ Dim_Driver ⨝ Dim_Sensor  

This lets us:
- Compare driver attributes by safety label (e.g. experience vs dangerous trips).
- Compare sensor trip features by safety label (e.g. speed/jerk vs label).

We will suffix overlapping columns to avoid collisions.


In [ ]:
# ================================
# 7. Join Fact_Label with Dimensions (Star-Join View)
# ================================

# Merge Fact_Label with Dim_Driver
fact_star_df = fact_label_df.merge(
    dim_driver_df,
    on="driver_key",
    how="left",
    suffixes=("", "_drv")
)

# Merge with Dim_Sensor
fact_star_df = fact_star_df.merge(
    dim_sensor_df,
    on="sensor_key",
    how="left",
    suffixes=("", "_sns")
)

print("\n=== fact_star_df preview ===")
print("fact_star_df shape:", fact_star_df.shape)
display(fact_star_df.head())

## 8. Dim_Sensor Features – Distributions by Safety Label

Here, we focus on a subset of important engineered features from `Dim_Sensor`:

- Speed behaviour: `FE_speed_mean`, `FE_speed_std`, `FE_speed_max`
- Acceleration & jerk: `FE_acc_mag_max`, `FE_jerk_p95`
- Turning behaviour: `FE_turn_delta_std_deg`, `FE_turn_rate_med_deg_s`

For each feature, we:
- Plot histogram split by `label` (safe vs dangerous).
- Plot boxplot by `label`.

This reveals whether dangerous trips tend to have higher speeds, more abrupt turns, or stronger acceleration spikes.


In [ ]:
# ================================
# 8. Dim_Sensor Features – Distributions by Safety Label
# ================================

sensor_feature_focus = [
    "FE_speed_mean",
    "FE_speed_std",
    "FE_speed_max",
    "FE_acc_mag_max",
    "FE_jerk_p95",
    "FE_turn_delta_std_deg",
    "FE_turn_rate_med_deg_s",
]

for col in sensor_feature_focus:
    if col in fact_star_df.columns:
        print(f"\n=== Sensor Feature: {col} (by label) ===")
        plot_numeric_distribution(
            fact_star_df,
            col=col,
            color_col=label_col,
            nbins=50,
            title_prefix="Fact+Dim: "
        )
    else:
        print(f"[WARN] Column {col} not found in fact_star_df.")

## 9. Outlier Analysis (IQR Method)

We now look for **outliers** in key numeric features using the standard IQR rule:

- Compute Q1 (25th percentile), Q3 (75th percentile), and IQR = Q3 − Q1.
- Define lower bound = Q1 − 1.5 × IQR, upper bound = Q3 + 1.5 × IQR.
- Values outside this range are flagged as potential outliers.

We do this for a focused set of features that are likely to affect safety labels:

- Driver side: `no_of_years_driving_exp`, `rating`, `car_model_year`.
- Sensor side: the `sensor_feature_focus` list from the previous section.


In [ ]:
# ================================
# 9. Outlier Analysis (IQR Method)
# ================================

driver_outlier_features = [
    "no_of_years_driving_exp",
    "rating",
    "car_model_year",
]

outlier_features = driver_outlier_features + sensor_feature_focus

rows = []
outlier_masks = {}

for col in outlier_features:
    if col not in fact_star_df.columns:
        print(f"[WARN] Column {col} not found in fact_star_df; skipping outlier analysis.")
        continue

    series = fact_star_df[col].dropna()
    if series.empty:
        print(f"[INFO] Column {col} has no non-null values; skipping.")
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask_outlier = (fact_star_df[col] < lower) | (fact_star_df[col] > upper)
    n_outliers = mask_outlier.sum()
    pct_outliers = (n_outliers / fact_star_df.shape[0]) * 100

    rows.append({
        "feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "n_outliers": n_outliers,
        "pct_outliers": round(pct_outliers, 2),
    })

    outlier_masks[col] = mask_outlier

outlier_summary_df = pd.DataFrame(rows).sort_values("pct_outliers", ascending=False)
print("\n=== IQR-based Outlier Summary (Fact + Dim) ===")
display(outlier_summary_df)

# Optional: inspect outlier rows for one feature
feature_to_inspect = "FE_speed_max"  # change to any feature in outlier_features

mask = outlier_masks.get(feature_to_inspect)
if mask is not None:
    print(f"\n=== Outlier rows for {feature_to_inspect} ===")
    cols_to_show = [label_col, feature_to_inspect]
    display(fact_star_df.loc[mask, cols_to_show].head(30))

    print(f"\nLabel distribution among outliers for {feature_to_inspect}:")
    display(fact_star_df.loc[mask, label_col].value_counts(normalize=True) * 100)
else:
    print(f"No outlier mask stored for {feature_to_inspect}.")

## 10. Label-wise Summary Ranges for Key Features

Finally, we summarise how the ranges of key features differ between:

- Safe trips (`label = 0`)
- Dangerous trips (`label = 1`)

We compute `.describe()` grouped by `label` for the most important driver and sensor features.  
This is useful for your report to show how behaviours (speed, jerk, turning, etc.) separate the two classes.


In [ ]:
# ================================
# 10. Label-wise Summary Ranges for Key Features
# ================================

label_feature_list = driver_outlier_features + sensor_feature_focus
available_features = [f for f in label_feature_list if f in fact_star_df.columns]

grouped_desc = fact_star_df.groupby(label_col)[available_features].describe().T
print("\n=== Label-wise descriptive statistics for key features ===")
display(grouped_desc)